In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from enum import Enum
import random
import string
import requests
from collections import defaultdict
import pandas as pd
from rapidfuzz import fuzz
import re
import time


In [ ]:
#### Helpers
class ErrorType(Enum):
    clean = 0
    no_author = 1
    typo_in_title = 2
    no_year = 3
    title_with_drop_word = 4


def remove_by_index(text, index_to_remove):
    return text[:index_to_remove] + text[index_to_remove + 1 :]


def extract_year_crossref(item):
    for field in ["published-print", "published-online", "issued"]:
        if field in item:
            return item[field]["date-parts"][0][0]

    return None


def generate_typo(line):
    count_of_typo = random.randint(1, 4)
    for _ in range(count_of_typo):
        type_of_typo = random.randint(0, 1)
        random_index = random.randint(0, len(line) - 1)
        if type_of_typo == 0:
            line = list(line)
            line[random_index] = random.choice(string.ascii_lowercase)
            line = "".join(line)
        else:
            line = remove_by_index(line, random_index)
    return line


def drop_word(line):
    words = line.split()
    words.pop(random.randint(0, len(words) - 1))
    return " ".join(words)


def generate_query(citation, type_of_error):
    if citation["source"] == "crossref":
        if type_of_error == ErrorType.no_year:
            return f"{citation['title']} {citation['first_author']}"
        if type_of_error == ErrorType.no_author:
            return f"{citation['title']} {citation['year']}"
        if type_of_error == ErrorType.title_with_drop_word:
            return f"{drop_word(citation['title'])} {citation['first_author']} {citation['year']}"
        if type_of_error == ErrorType.typo_in_title:
            return f"{generate_typo(citation['title'])} {citation['first_author']} {citation['year']}"
        return f"{citation['title']} {citation['first_author']} {citation['year']}"
    else:
        if type_of_error == ErrorType.typo_in_title:
            return f"{generate_typo(citation['title'])}"
        if type_of_error == ErrorType.title_with_drop_word:
            return f"{drop_word(citation['title'])}"
        return f"{citation['title']}"


In [ ]:
#Main Variables
error_type_strings = {
    ErrorType.no_author: "No author error",
    ErrorType.title_with_drop_word: "Title with dropped word error",
    ErrorType.no_year: "No year error",
    ErrorType.typo_in_title: "Typo in title error",
    ErrorType.clean: "No errors",
}

results = []
ranking_rows = []


In [ ]:
##### Data retrieve
def process_authors(author_list):
    return [
        f"{author.get('given', '')} {author.get('family', '')}"
        for author in author_list
    ]


def extract_first_author(item):
    authors = item.get("author")

    if not authors:
        return None

    first = authors[0]
    given = first.get("given", "")
    family = first.get("family", "")

    full_name = f"{given} {family}".strip()

    return full_name if full_name else None


def get_articles_crossref(article_count):
    response = requests.get(
        f"https://api.crossref.org/works?filter=type:journal-article,has-references:true&rows={article_count}&select=DOI,title,author,container-title,issued,published-print,published-online",
    )
    items = response.json()["message"]["items"]
    for item in items:
        if "author" in item.keys():
            item["author"] = process_authors(item["author"])
        else:
            item["author"] = None
    return clean_citations(items)


def get_articles_cyberleninka(article_count):
    params = {
        "mode": "articles",
        "q": "",
        "size": article_count,
        "from": 0,
    }
    response = requests.post(
        "https://cyberleninka.ru/api/search", json=params, timeout=20
    )
    if response.status_code != 200:
        print("Crossref error:", response.status_code, response.text[:200])
        return -1, []
    try:
        data = response.json()['articles']
    except Exception:
        print("Not JSON response:", response.status_code, response.text[:200])
        return -1, []
    cleaned_data = []
    for item in data:
        cleaned_data.append(
            {
                "id": item["link"],
                "title": item["name"],
                "journal": item["journal"],
                "first_author": item["authors"][0],
                "year": item["year"],
                "source": "cyberleninka",
            }
        )
    return cleaned_data


def clean_citations(citations):
    cleaned = []
    for citation in citations:
        if "author" in citation.keys() and citation["title"] != ["In Response"]:
            cleaned.append(
                {
                    "id": citation["DOI"],
                    "title": citation["title"][0],
                    "year": citation["issued"]["date-parts"][0][0],
                    "first_author": citation["author"][0],
                    "journal": citation["container-title"][0],
                    "source": "crossref",
                }
            )
    return cleaned


In [ ]:
def check_citation_match_crossref(query, citation):
    params = {
        "query.bibliographic": query,
        "rows": "10",
    }
    candidates = []
    response = requests.get(
        "https://api.crossref.org/works",
        params=params,
        timeout=20,
        headers={"User-Agent": "citation-matcher/0.1 (mailto:your_email@example.com)"},
    )

    if response.status_code != 200:
        print("Crossref error:", response.status_code, response.text[:200])
        return -1, []
    try:
        data = response.json()
    except Exception:
        print("Not JSON response:", response.status_code, response.text[:200])
        return -1, []
    possible_mathces = data["message"]["items"]
    match_rank = -1
    rank = 1
    for match in possible_mathces:
        label = 0
        if match["DOI"] == citation["id"]:
            label = 1
            match_rank = rank
        candidates.append(
            {
                "query": query,
                "candidate_id": match["DOI"],
                "label": label,
                "candidate_year": extract_year_crossref(match),
                "candidate_title": match.get("title", [""])[0],
                "true_id": citation["id"],
                "candidate_rank": rank,
                "candidate_author": extract_first_author(match),
                "query_author": citation["first_author"],
            }
        )
        rank += 1
    return match_rank, candidates


def check_citation_match_cyberleninka(query, citation):
    params = {
        "mode": "articles",
        "q": query,
        "size": 10,
        "from": 0,
    }
    candidates = []
    response = requests.post(
        "https://cyberleninka.ru/api/search", json=params, timeout=20
    )
    if response.status_code != 200:
        print("Crossref error:", response.status_code, response.text[:200])
        return -1, []
    try:
        data = response.json()
    except Exception:
        print("Not JSON response:", response.status_code, response.text[:200])
        return -1, []
    if "articles" not in data:
        print("CyberLeninka response without articles:", data)
        return -1, []
    possible_matches = data.get("articles", [])
    match_rank = -1
    rank = 1
    for match in possible_matches[:10]:
        label = 0
        if match["link"] == citation["id"]:
            label = 1
            match_rank = rank
        candidates.append(
            {
                "query": query,
                "label": label,
                "true_id": citation["id"],
                "candidate_rank": rank,
                "candidate_id": match.get("id"),
                "candidate_year": match.get("link"),
                "candidate_title": match.get("name", ""),
                "candidate_author": (
                    match.get("authors", [None])[0] if match.get("authors") else None
                ),
                "query_author": citation.get("first_author"),
                "journal": match.get("journal"),
            }
        )
        rank += 1
    time.sleep(0.5)

    return match_rank, candidates


def process_one(article, type_of_error):
    query = generate_query(article, type_of_error)
    if article["source"] == "crossref":
        match_rank, candidates = check_citation_match_crossref(query, article)
    elif type_of_error == ErrorType.no_author or type_of_error == ErrorType.no_year:
        return {}
    else:
        match_rank, candidates = check_citation_match_cyberleninka(query, article)

    return {
        "Original Citation": article,
        "Query": query,
        "Type of Error": error_type_strings[type_of_error],
        "Match Number": match_rank,
        "Candidates": candidates,
    }


#### Main Flow
def main(type_of_error: ErrorType, articles):
    global results, ranking_rows
    matches = defaultdict(int)
    count = 0
    for article in articles:
        result = process_one(article, type_of_error=type_of_error)

        if result == {}:
            continue

        results.append(result)
        ranking_rows.extend(result["Candidates"])
        matches[result["Match Number"]] += 1

        # print("Processing citation:", result["Query"])
        # print(
        #     f"Match is found in top #{result['Match Number']}"
        #     if result["Match Number"] > 0
        #     else "Nothing's found in top 10"
        # )
        count += 1
        if count % 10 == 0:
            print(f"Processed {count} articles")

    # with ThreadPoolExecutor(max_workers=1) as executor:
    #     futures = [
    #         executor.submit(process_one, article, type_of_error) for article in articles
    #     ]
    #     for future in as_completed(futures):
    #         result = future.result()
    #         if result != {}:
    #             results.append(result)
    #         ranking_rows.extend(result["Candidates"])
    #         matches[result["Match Number"]] += 1

    #         print("Processing citation:", result["Query"])
    #         print(
    #             f"Match is found in top #{result['Match Number']}"
    #             if result["Match Number"] > 0
    #             else "Nothing's found in top 10"
    #         )

    print("SUMMARY:")
    print("=" * 50)
    print(f"Results for error {error_type_strings[type_of_error]}")
    for i in range(1, 11):
        print(f"Match in top #{i}: \t\t {matches[i]}")
    print("\n\n", "Not found in top #10:", matches[-1])


if __name__ == "__main__":
    articles = get_articles_cyberleninka(50) + get_articles_crossref(50)
    main(ErrorType.clean, articles)
    main(ErrorType.typo_in_title, articles.copy())
    main(ErrorType.no_author, articles)
    main(ErrorType.no_year, articles)
    main(ErrorType.title_with_drop_word, articles)
    df = pd.DataFrame(results)
    df.to_csv("../data/results.csv", index=False)
    df["top1"] = df["Match Number"] == 1
    df["top3"] = df["Match Number"].between(1, 3)
    df["top10"] = df["Match Number"].between(1, 10)
    summary = df.groupby("Type of Error").agg(
        top1=("top1", "mean"),
        top3=("top3", "mean"),
        top10=("top10", "mean"),
        count=("Match Number", "count"),
    )
    summary.to_csv("../data/summary.csv")
    ranking_df = pd.DataFrame(ranking_rows)
    ranking_df.to_csv("../data/ranking_dataset.csv", index=False)
    print(summary)


In [ ]:
articles = get_articles_crossref(2)
print(articles[0])

articles = get_articles_cyberleninka(2)
print(articles[0])

{'id': '10.17116/rosstomat20251801140', 'title': 'Substantiation of the anti-inflammatory effect of photodynamic therapy in periodontal diseases', 'year': 2025, 'first_author': 'O.N. Risovannaya', 'journal': 'Russian Journal of  Stomatology', 'source': 'crossref'}
{'id': '/article/n/publichnyy-dogovor-kak-element-zaschity-slaboy-storony', 'title': 'Публичный договор как элемент защиты слабой стороны', 'journal': 'Юридическая наука', 'first_author': 'Каширин Игорь Олегович', 'year': 2012, 'source': 'cyberleninka'}


In [ ]:
df = pd.read_csv("../data/ranking_dataset.csv")
df["title_similarity"] = df.apply(
    lambda row: fuzz.token_sort_ratio(str(row["query"]), str(row["candidate_title"])),
    axis=1,
)
df["title_token_set_similarity"] = df.apply(
    lambda row: fuzz.token_set_ratio(str(row["query"]), str(row["candidate_title"])),
    axis=1,
)


def extract_year_from_query(query):
    match = re.search(r"\b(19|20)\d{2}\b", str(query))
    if match:
        return int(match.group())
    return None


df["query_year"] = df["query"].apply(extract_year_from_query)


In [ ]:
df["year_difference"] = abs(df["candidate_year"] - df["query_year"])
df["first_author_similarity"] = df.apply(
    lambda row: fuzz.token_sort_ratio(
        str(row["query_author"]),
        str(row["candidate_author"])
    ),
    axis=1,
)

In [ ]:
df.groupby("label")["year_difference"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,1703.0,7.958896,12.409227,0.0,1.0,4.0,10.0,122.0
1,194.0,1.067010,7.933705,0.0,0.0,0.0,0.0,71.0


In [ ]:

df.to_csv("../data/ranking_dataset_with_features-v3.csv", index=False)

In [ ]:
df.groupby("label")["first_author_similarity"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,2974.0,21.569589,21.607966,0.0,0.0,21.052632,29.62963,100.0
1,337.0,69.732938,46.009703,0.0,0.0,100.000000,100.00000,100.0
